# Notebook 07 — PPI Network Analysis

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 07 of 12  
**Time estimate:** 75 minutes

> Protein–protein interactions are the physical wiring of the cell.
> Analyzing this network reveals which proteins are central, which pathways
> are modular, and how diseases perturb the interactome.

---
## Step 1 — Motivation

PPI networks are the prototypical biological network — they connect all concepts from
NB02–NB06 to real biological data. This notebook applies the full analysis toolkit
(topology, centrality, community detection) to a realistic PPI network, building the
analysis pipeline that the mini-project will require.

---
## Step 2 — Intuition

A PPI network tells us which proteins physically bind. But "binding" is not static:
- Some interactions are constitutive (always present in the same complex)
- Some are transient (signaling — brief phosphorylation-dependent contacts)
- Some are cell-type specific

The interactome is therefore an *overlay* of many context-specific networks.
Despite this, the aggregate (all-cell-type) PPI network has consistent scale-free
topology, short paths, and modular structure — suggesting universal organizing principles.

---
## Step 3 — Biological Background

**Data sources and methods:**
- **Yeast two-hybrid (Y2H):** screen for pairwise physical interactions in vivo; high throughput, higher false positive rate
- **Affinity purification + mass spectrometry (AP-MS):** pull down protein complexes; identifies co-complex interactions; lower noise
- **Co-fractionation MS:** native complexes separated by size/charge; interactions inferred from co-elution
- **Literature curation:** manually extracted from published experiments (BioGRID, IntAct)

**Confidence scoring (STRING):**
- STRING integrates multiple evidence types: experimental, co-expression, text mining, genomic context
- Each pair gets a combined confidence score 0–1000
- Common threshold: ≥700 ("high confidence")

**Interactome completeness:**
- The human interactome is estimated to contain ~300,000–500,000 binary interactions
- Current databases cover ~100,000–150,000 — roughly 25–40% complete
- Biased toward well-studied proteins (sampling bias affects topology analysis)

**Key PPI hubs (human):**
- TP53 (p53): ~300 known interactions — DNA damage response, apoptosis, transcription
- UBC (ubiquitin): ~1000 interactions — ubiquitination of virtually every pathway
- EGFR: ~200 interactions — receptor tyrosine kinase, growth signaling
- These are both hubs and therapeutic targets

---
## Step 4 — Computational Explanation

**Full PPI analysis pipeline:**
1. Load edge list → build graph
2. Extract largest connected component (LCC)
3. Basic statistics: N, M, mean degree, diameter (on sample)
4. Degree distribution: fit power-law exponent
5. Clustering and path length
6. Centrality: degree, betweenness, PageRank
7. Community detection: Louvain → modularity Q
8. Interpretation: top hubs, top bottlenecks, community sizes

We use a **synthetic 300-node PPI** in this notebook (DS01 from datasets/README.md).
Assignment A01 applies the same pipeline to real STRING data.

In [ ]:
# Step 6 — Python: Full PPI analysis pipeline on synthetic network
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict, Counter

# ---- Generate synthetic PPI: SBM modules + BA hub structure ----
def generate_synthetic_ppi(n_per_module=60, n_modules=4, p_in=0.15, p_out=0.01,
                              n_extra_hubs=10, seed=42):
    """
    Create a biologically realistic synthetic PPI:
    - Base: Stochastic Block Model (functional modules)
    - Extra: hub nodes connected to random nodes (preferential attachment)
    """
    rng = np.random.default_rng(seed)
    n = n_per_module * n_modules
    adj = {i: set() for i in range(n)}
    true_labels = np.repeat(np.arange(n_modules), n_per_module)
    
    # SBM edges
    for i in range(n):
        for j in range(i+1, n):
            p = p_in if true_labels[i] == true_labels[j] else p_out
            if rng.random() < p:
                adj[i].add(j); adj[j].add(i)
    
    # Add hub nodes: each hub connects to 20 random nodes
    protein_names = [f'P{i:04d}' for i in range(n)]
    hub_labels = []
    for h in range(n_extra_hubs):
        hub_id = n + h
        adj[hub_id] = set()
        true_labels = np.append(true_labels, -1)  # hubs not in any module
        hub_labels.append(hub_id)
        # Connect hub to random nodes, with some bias toward high-degree nodes
        degrees = np.array([len(adj[v]) for v in range(n)])
        probs = (degrees + 1.0) / (degrees + 1.0).sum()
        targets = rng.choice(n, size=min(25, n), replace=False, p=probs)
        for t in targets:
            adj[hub_id].add(t); adj[t].add(hub_id)
    
    protein_names += [f'HUB{i}' for i in range(n_extra_hubs)]
    n_total = n + n_extra_hubs
    return adj, true_labels, protein_names, n_total

adj_ppi, true_labels, protein_names, n_total = generate_synthetic_ppi()

# Build NetworkX graph
G = nx.Graph()
for v in adj_ppi:
    for u in adj_ppi[v]:
        if u > v:
            G.add_edge(v, u)

print(f'Network: N={G.number_of_nodes()}, E={G.number_of_edges()}')
print(f'Density: {nx.density(G):.4f}')
degrees = np.array([d for _, d in G.degree()])
print(f'Mean degree: {degrees.mean():.2f}')
print(f'Max degree (largest hub): {degrees.max()}')

# ---- Largest connected component ----
lcc_nodes = max(nx.connected_components(G), key=len)
G_lcc = G.subgraph(lcc_nodes).copy()
print(f'\nLCC: {G_lcc.number_of_nodes()} nodes ({100*G_lcc.number_of_nodes()/G.number_of_nodes():.1f}%)')

# ---- Centrality ----
deg_cent = nx.degree_centrality(G_lcc)
btwn_cent = nx.betweenness_centrality(G_lcc, normalized=True)
page_rank = nx.pagerank(G_lcc, alpha=0.85)

# Top hubs and bottlenecks
top_hubs = sorted(deg_cent, key=deg_cent.get, reverse=True)[:10]
top_btwn = sorted(btwn_cent, key=btwn_cent.get, reverse=True)[:10]
print(f'\nTop 5 hubs (degree): {[f"{protein_names[v]} ({deg_cent[v]:.3f})" for v in top_hubs[:5]]}')
print(f'Top 5 bottlenecks (betweenness): {[f"{protein_names[v]} ({btwn_cent[v]:.3f})" for v in top_btwn[:5]]}')

# ---- Community detection ----
try:
    from networkx.algorithms.community import louvain_communities
    comms = louvain_communities(G_lcc, seed=42)
except Exception:
    comms = nx.algorithms.community.greedy_modularity_communities(G_lcc)

comm_labels = np.zeros(G.number_of_nodes(), dtype=int) - 1
for cid, comm in enumerate(comms):
    for node in comm:
        comm_labels[node] = cid

Q = nx.algorithms.community.quality.modularity(G_lcc, comms)
print(f'\nCommunities detected: {len(comms)}')
print(f'Community sizes: {sorted([len(c) for c in comms], reverse=True)[:8]}')
print(f'Modularity Q: {Q:.4f}')

In [ ]:
# Step 7 — Visualization: 4-panel PPI analysis figure
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

lcc_node_list = sorted(G_lcc.nodes())
cmap = plt.cm.get_cmap('tab10', max(comm_labels) + 2)

# Panel A: Network (spring layout, color=community, size=degree)
ax = axes[0, 0]
try:
    pos = nx.spring_layout(G_lcc, seed=42, k=0.5)
except Exception:
    pos = nx.circular_layout(G_lcc)

nx.draw_networkx_edges(G_lcc, pos, ax=ax, alpha=0.2, width=0.5, edge_color='grey')

node_colors = [cmap(comm_labels[v]) if comm_labels[v] >= 0 else 'black' for v in lcc_node_list]
node_sizes = [20 + 300 * deg_cent[v] for v in lcc_node_list]

nx.draw_networkx_nodes(G_lcc, pos, ax=ax,
                         nodelist=lcc_node_list,
                         node_color=node_colors,
                         node_size=node_sizes, alpha=0.8)
ax.set_title('A. PPI Network\n(color=community, size=degree)', fontsize=10)
ax.axis('off')

# Panel B: Degree distribution (log-log)
ax = axes[0, 1]
lcc_degrees = np.array([G_lcc.degree(v) for v in lcc_node_list])
k_vals, k_counts = np.unique(lcc_degrees, return_counts=True)
k_prob = k_counts / len(lcc_degrees)
ax.scatter(k_vals, k_prob, color='steelblue', s=30, alpha=0.8, zorder=3)
ax.set_xscale('log'); ax.set_yscale('log')
# Power-law fit
mask = k_vals >= 2
if mask.sum() > 2:
    log_k = np.log(k_vals[mask])
    log_p = np.log(k_prob[mask])
    coeffs = np.polyfit(log_k, log_p, 1)
    gamma_fit = -coeffs[0]
    k_fit = np.exp(np.linspace(log_k.min(), log_k.max(), 50))
    p_fit = np.exp(coeffs[1]) * k_fit**coeffs[0]
    ax.plot(k_fit, p_fit, 'r--', lw=1.5, label=f'Power-law γ={gamma_fit:.2f}')
    ax.legend(fontsize=8)
ax.set_xlabel('Degree k')
ax.set_ylabel('P(k)')
ax.set_title('B. Degree distribution (log-log)', fontsize=10)

# Panel C: Hub vs. bottleneck centrality
ax = axes[1, 0]
deg_arr = np.array([deg_cent.get(v, 0) for v in lcc_node_list])
btwn_arr = np.array([btwn_cent.get(v, 0) for v in lcc_node_list])
pr_arr = np.array([page_rank.get(v, 0) for v in lcc_node_list])
sc = ax.scatter(deg_arr, btwn_arr, c=pr_arr, cmap='viridis', s=25, alpha=0.7)
plt.colorbar(sc, ax=ax, label='PageRank')
# Annotate top 5 bottlenecks
for v in top_btwn[:5]:
    ax.annotate(protein_names[v], (deg_cent[v], btwn_cent[v]), fontsize=6, alpha=0.7)
ax.set_xlabel('Degree centrality')
ax.set_ylabel('Betweenness centrality')
ax.set_title('C. Hub vs. Bottleneck\n(color = PageRank)', fontsize=10)

# Panel D: Community size distribution
ax = axes[1, 1]
comm_sizes = sorted([len(c) for c in comms], reverse=True)
ax.bar(range(len(comm_sizes)), comm_sizes, color='steelblue', edgecolor='black')
ax.set_xlabel('Community rank')
ax.set_ylabel('Community size')
ax.set_title(f'D. Community sizes\nQ={Q:.3f}, {len(comms)} communities detected', fontsize=10)

plt.suptitle('Module 12 NB07: PPI Network Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ppi_network_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Save the synthetic PPI as a CSV edge list to `datasets/ppi_synthetic.csv`.
   Reload it from the file and verify N, M, and mean degree are identical.
2. Identify which detected communities overlap most with the planted SBM modules.
   Compute the Adjusted Rand Index.
3. The hub nodes (HUB0–HUB9) are added with higher connectivity. Do they appear in
   the same community? What does this suggest about how community detection handles
   "cross-module" hub proteins?
4. (Optional real data) Download the STRING human PPI (score ≥ 700) following
   `datasets/README.md DS02`. Apply the same 4-panel analysis pipeline.

---
## Step 10 — Quiz

1. What is the difference between Y2H and AP-MS as PPI detection methods?
2. Why is the human interactome estimated to be only ~25–40% complete?
3. What is sampling bias in PPI networks, and how does it affect topology analysis?
4. What is the STRING confidence score? What threshold is commonly used?
5. What does it mean for a protein to be both a hub and a bottleneck?

---
## Step 12 — Reflection

> *[In one paragraph: describe what you would expect to find if you restricted the
> PPI network to interactions measured only in a single cell type (e.g., neurons).
> How would the topology differ from the aggregate interactome?]*

---
*Next: `08_gene_regulatory_networks_boolean_models.ipynb`*